Testing Multi-channel classification

In [ ]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras import layers, Model, Input
from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

matplotlib.use('QtAgg') # for GUI 
mne.set_log_level("CRITICAL")

In [ ]:
# starting with basic CNN model 
def CNN_model(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer
    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)


    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)

    x = Flatten()(x)

    # two heads for each label 
    # output head for Zygo
    out_zygo = Dense(num_classes, activation='softmax', name="zygo_output")(x)

    # output head for Corr
    out_corr = Dense(num_classes, activation='softmax', name="corr_output")(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])

    return model  # Return the compiled model

Loading in Data

In [ ]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

# dummy column of duration 
#features_all = features_all.assign(Duration_zygo = np.random.randint(0, 6, size=np.shape(features_all)[0]))
#features_all = features_all.assign(Duration_corr = np.random.randint(0, 6, size=np.shape(features_all)[0]))


In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])


''' 
y_durations = np.transpose([features_all_temp["Duration_zygo"].astype(int).to_numpy(),
                    features_all_temp["Duration_corr"].astype(int).to_numpy()])
'''

# y = np.column_stack((y_contractions, y_durations)) if doing durations 

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]
 

In [ ]:
num_classes = len(np.unique(y))    
print(num_classes)

In [ ]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model = CNN_model(input_shape, num_classes,feature_num)
    model.compile(optimizer='adam', loss=["sparse_categorical_crossentropy", "sparse_categorical_crossentropy"], metrics=['accuracy','accuracy'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]])) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model.evaluate(X_test,[y_test[:, 0], y_test[:, 1]])
    print(scores)
    cvScores.append(scores[1] * 100)

    k += 1 
    

model_history = model.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]])) #, callbacks=[early_stop])

In [64]:
print(scores)

[1.0954093933105469, 0.45893749594688416, 0.6364719271659851, 0.9368055462837219, 0.9430555701255798]


In [66]:
model_history = model.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]])) #, callbacks=[early_stop])

Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 70ms/step - corr_output_accuracy: 0.8156 - corr_output_loss: 0.9350 - loss: 1.9412 - zygo_output_accuracy: 0.8174 - zygo_output_loss: 1.0062 - val_corr_output_accuracy: 0.8792 - val_corr_output_loss: 0.5714 - val_loss: 1.0956 - val_zygo_output_accuracy: 0.8694 - val_zygo_output_loss: 0.5242
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 12s 66ms/step - corr_output_accuracy: 0.8823 - corr_output_loss: 0.4367 - loss: 0.8674 - zygo_output_accuracy: 0.8743 - zygo_output_loss: 0.4306 - val_corr_output_accuracy: 0.8847 - val_corr_output_loss: 0.5783 - val_loss: 1.1121 - val_zygo_output_accuracy: 0.8792 - val_zygo_output_loss: 0.5338
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - corr_output_accuracy: 0.9260 - corr_output_loss: 0.2506 - loss: 0.4976 - zygo_output_accuracy: 0.9198 - zygo_output_loss: 0.2470 - val_corr_output_accuracy: 0.8792 - val_corr_output_loss: 0.7246 - val_loss: 1.4422 - val_zygo_output_accuracy: 0.8799 - val_zygo_output_loss:

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

In [67]:
# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model.predict(X_test)

# Convert probabilities to class labels
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)

# true labels for corr and zygo 
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]
y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# calculate accuracy for each muscle group 
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)

# calculate f1 score for each muscle group 
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')

print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n -------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)

# average score
print("\n -------- Average --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)  

180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
-------- Zygo --------
Training Accuracy: 0.9977430555555555
Test Accuracy: 0.8868055555555555
Training F1 Score: 0.9977386955561075
Test F1 Score: 0.8625500211177448

 -------- Corr --------
Training Accuracy: 0.9996527777777777
Test Accuracy: 0.8805555555555555
Training F1 Score: 0.9996527777777777
Test F1 Score: 0.8560035739411325

 -------- Average --------
Training Accuracy: 0.9986979166666666
Test Accuracy: 0.8836805555555556
Training F1 Score: 0.9986957366669427
Test F1 Score: 0.8592767975294386
